In [0]:
import csv, io, os, sys, random
from datetime import date, timedelta

dbutils.widgets.text("start_date", "2026-11-01")
dbutils.widgets.text("n_days", "3")
start = date.fromisoformat(dbutils.widgets.get("start_date"))
n_days = int(dbutils.widgets.get("n_days"))

repo_root = os.path.dirname(os.getcwd())          # notebook лежить у notebooks/
sys.path.insert(0, repo_root)
from src.generator import TOURS, generate_events, add_data_issues, generate_account_changes

landing = "/Volumes/workspace/product_tours/landing"

def to_csv(rows):
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
    return buf.getvalue()

dbutils.fs.put(f"{landing}/tours/tours.csv", to_csv(TOURS), overwrite=True)

for i in range(n_days):
    day = start + timedelta(days=i)
    rnd = random.Random(day.toordinal())           # seed = дата → той самий день завжди дає ті самі дані
    lines = add_data_issues(generate_events(day, n_sessions=300, rnd=rnd), i, rnd)
    dbutils.fs.put(f"{landing}/events/events_{day}.jsonl", "\n".join(lines), overwrite=True)
    dbutils.fs.put(f"{landing}/accounts/accounts_{day}.csv",
                   to_csv(generate_account_changes(day, i, rnd)), overwrite=True)
    print(f"{day}: {len(lines)} event lines")